# Setup

In [1]:
# Tìm thư mục gốc của dự án để notebook biết nơi chứa mã nguồn và dữ liệu.
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment

def find_project_root(max_up=6):
    p = Path.cwd()
    for _ in range(max_up + 1):
        if (p / 'src').exists() and (p / 'configs').exists(): return p
        if p.parent == p: break
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()
sys.path.append(str(PROJECT_ROOT))

from src.config import load_yaml
from src.io_utils import save_csv, save_numpy

pipeline_cfg = load_yaml(PROJECT_ROOT / 'configs/pipeline_config.yaml')
assoc_cfg = pipeline_cfg['association']

tracklet_meta = pd.read_csv(PROJECT_ROOT / 'outputs/embeddings/tracklet_embeddings_meta.csv')
tracklet_embs = np.load(PROJECT_ROOT / 'outputs/embeddings/tracklet_embeddings.npy')

tracklet_meta.head()

,tracklet_embedding_index,camera_id,local_track_id,num_crops,mean_quality_score
0,0,MOT17-02-FRCNN,1,5,0.982400
1,1,MOT17-02-FRCNN,2,5,0.950629
2,2,MOT17-02-FRCNN,3,5,0.846067
3,3,MOT17-02-FRCNN,4,5,1.022948
4,4,MOT17-02-FRCNN,5,5,0.603006


# Realistic Simulation multi-camera

Nâng cấp simulation với:
- Multi-difficulty noise (easy/medium/hard)
- Partial occlusion simulation
- Tỷ lệ xuất hiện ở quầy vé theo phân bố thực tế

In [2]:
# Tạo dữ liệu giả lập nâng cao để mô phỏng kết nối giữa hai camera.
np.random.seed(42)

sim_meta = tracklet_meta.copy()

# Giả lập tỷ lệ có vé: phân bố thực tế hơn
# Phần lớn du khách (60%) mua vé; một số (40%) có thể là dân địa phương hoặc chưa mua
sim_meta['appears_at_ticket'] = np.random.rand(len(sim_meta)) < 0.60

entrance_meta = sim_meta.copy()
entrance_meta['sim_camera_id'] = 'cam_entrance_01'

ticket_meta = sim_meta[sim_meta['appears_at_ticket']].copy()
ticket_meta['sim_camera_id'] = 'cam_ticket_01'

def realistic_camera_transform(emb, difficulty='medium'):
    """Mô phỏng sự khác biệt embedding giữa hai camera với nhiều mức độ."""
    if difficulty == 'easy':
        noise_std = 0.02  # cùng điều kiện ánh sáng, góc tương tự
    elif difficulty == 'medium':
        noise_std = 0.05  # khác góc camera, ánh sáng khác
    else:  # hard
        noise_std = 0.08
        # Partial occlusion: mask random 15% features
        mask = np.random.rand(len(emb)) > 0.15
        emb = emb * mask
    
    emb2 = emb + np.random.normal(0, noise_std, size=emb.shape)
    norm = np.linalg.norm(emb2)
    return emb2 / norm if norm > 0 else emb2

# Gán difficulty dựa trên đặc điểm tracklet
ticket_embeddings = []
difficulty_labels = []
for _, row in ticket_meta.iterrows():
    idx = int(row['tracklet_embedding_index'])
    emb = tracklet_embs[idx].copy()
    
    # Chọn difficulty dựa trên quality score (proxy cho điều kiện thực tế)
    q = row.get('mean_quality_score', 0.5)
    if q >= 0.7: diff = 'easy'
    elif q >= 0.4: diff = 'medium'
    else: diff = 'hard'
    
    emb2 = realistic_camera_transform(emb, difficulty=diff)
    ticket_embeddings.append(emb2)
    difficulty_labels.append(diff)

ticket_embeddings = np.vstack(ticket_embeddings) if ticket_embeddings else np.empty((0, tracklet_embs.shape[1]))
ticket_meta = ticket_meta.copy()
ticket_meta['sim_difficulty'] = difficulty_labels

print('Entrance tracklets:', len(entrance_meta))
print('Ticket tracklets:', len(ticket_meta))
print('Difficulty distribution:', pd.Series(difficulty_labels).value_counts().to_dict())

Entrance tracklets: 364
Ticket tracklets: 218
Difficulty distribution: {'easy': 171, 'medium': 47}


# Cosine similarity + Hungarian matching

In [3]:
# So sánh đặc trưng embedding giữa hai camera để tìm cặp tracklet giống nhau.
similarity_threshold = assoc_cfg['similarity_threshold']

entrance_embs = tracklet_embs[entrance_meta['tracklet_embedding_index'].values]

sim_matrix = cosine_similarity(entrance_embs, ticket_embeddings)

cost_matrix = 1.0 - sim_matrix

row_ind, col_ind = linear_sum_assignment(cost_matrix)

association_rows = []

for r, c in zip(row_ind, col_ind):
    sim_score = float(sim_matrix[r, c])

    if sim_score >= similarity_threshold:
        source = entrance_meta.iloc[r]
        target = ticket_meta.iloc[c]

        association_rows.append({
            'source_camera': 'cam_entrance_01',
            'source_tracklet_id': int(source['local_track_id']),
            'target_camera': 'cam_ticket_01',
            'target_tracklet_id': int(target['local_track_id']),
            'reid_score': sim_score,
            'time_score': 1.0,
            'quality_score': float(source['mean_quality_score']),
            'total_score': sim_score,
            'matched': True,
            'sim_difficulty': target.get('sim_difficulty', 'medium')
        })

assoc_df = pd.DataFrame(association_rows)

out_path = PROJECT_ROOT / 'outputs/associations/associations.csv'
save_csv(assoc_df, out_path)

print('Saved:', out_path)
print('Total associations:', len(assoc_df))
if 'sim_difficulty' in assoc_df.columns and len(assoc_df) > 0:
    print('Matches by difficulty:', assoc_df['sim_difficulty'].value_counts().to_dict())
assoc_df.head()

Saved: D:\hoiancomputervision\ticket_suspicion_project\outputs\associations\associations.csv
Total associations: 218
Matches by difficulty: {'easy': 171, 'medium': 47}


,source_camera,source_tracklet_id,target_camera,target_tracklet_id,reid_score,time_score,quality_score,total_score,matched,sim_difficulty
0,cam_entrance_01,1,cam_ticket_01,1,0.913689,1.0,0.982400,0.913689,True,easy
1,cam_entrance_01,4,cam_ticket_01,4,0.917324,1.0,1.022948,0.917324,True,easy
2,cam_entrance_01,5,cam_ticket_01,5,0.685172,1.0,0.603006,0.685172,True,medium
3,cam_entrance_01,6,cam_ticket_01,6,0.670481,1.0,0.646009,0.670481,True,medium
4,cam_entrance_01,7,cam_ticket_01,7,0.916015,1.0,0.734415,0.916015,True,easy


# Global temporary ID

In [4]:
# Gắn global_id cho mỗi người dựa trên kết quả ghép tracklet giữa hai camera.
global_rows = []

for i, row in entrance_meta.reset_index(drop=True).iterrows():
    local_id = int(row['local_track_id'])

    matched = assoc_df[assoc_df['source_tracklet_id'] == local_id]

    if len(matched) > 0:
        ticket_evidence = True
        target_tracklet = int(matched.iloc[0]['target_tracklet_id'])
        reid_score = float(matched.iloc[0]['reid_score'])
    else:
        ticket_evidence = False
        target_tracklet = None
        reid_score = None

    global_rows.append({
        'global_id': f'GID_{i:05d}',
        'entry_camera': 'cam_entrance_01',
        'entry_tracklet_id': local_id,
        'ticket_camera': 'cam_ticket_01' if ticket_evidence else None,
        'ticket_tracklet_id': target_tracklet,
        'ticket_evidence': bool(ticket_evidence),
        'reid_score': reid_score,
        'entry_time_sec': 0.0
    })

global_df = pd.DataFrame(global_rows)

out_path = PROJECT_ROOT / 'outputs/global_ids/global_ids.csv'
save_csv(global_df, out_path)

print('Saved:', out_path)
print('Total global IDs:', len(global_df))
print('With ticket evidence:', global_df['ticket_evidence'].sum())
global_df.head()

Saved: D:\hoiancomputervision\ticket_suspicion_project\outputs\global_ids\global_ids.csv
Total global IDs: 364
With ticket evidence: 219


,global_id,entry_camera,entry_tracklet_id,ticket_camera,ticket_tracklet_id,ticket_evidence,reid_score,entry_time_sec
0,GID_00000,cam_entrance_01,1,cam_ticket_01,1.0,True,0.913689,0.0
1,GID_00001,cam_entrance_01,2,None,NaN,False,NaN,0.0
2,GID_00002,cam_entrance_01,3,None,NaN,False,NaN,0.0
3,GID_00003,cam_entrance_01,4,cam_ticket_01,4.0,True,0.917324,0.0
4,GID_00004,cam_entrance_01,5,cam_ticket_01,5.0,True,0.685172,0.0
